# MMD-GAN sur CelebA — Étude comparative complète

**Référence :** Bińkowski, Sutherland, Arbel, Gretton — *Demystifying MMD GANs*, ICLR 2018.

Ce notebook regroupe **toutes les expériences** demandées dans une grille d'ablation unique :

| Noyau \\ Régularisation | sans GP | GP seul | GP + activation penalty |
|---|---|---|---|
| **RBF fixe** (σ=1) | exp 1 | exp 2 | exp 3 |
| **RQ\*** (rationnel quadratique + linéaire) | exp 4 | exp 5 | exp 6 |
| **Learnable RQ\*** (α appris) | exp 7 | exp 8 | exp 9 |

Chaque expérience **réinitialise** le générateur, le critique (et le noyau si appris) pour
une comparaison équitable. Les résultats sont stockés dans `RESULTS` puis comparés à la fin.


## 1. Imports et configuration

In [ ]:
import os
import math
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
@dataclass
class Config:
    # Données
    img_size: int = 64
    channels: int = 3
    batch_size: int = 64

    # Architectures
    z_dim: int = 100
    feat_dim: int = 16          # top-layer du critique (papier : 16 pour MMD)
    g_filters: int = 64
    d_filters: int = 16         # small critic (cf. papier)

    # MMD / kernel
    sigma: float = 1.0          # bandwidth du RBF fixe
    lambda_gp: float = 1.0      # poids du gradient penalty
    lambda_act: float = 1.0     # poids de la pénalité d'activation (note 19 du papier)

    # Optimisation
    lr: float = 1e-4
    beta1: float = 0.5
    beta2: float = 0.9
    n_critic: int = 5
    n_iters: int = 30000        # mets une valeur plus petite (ex. 5000) pour un test rapide
    log_every: int = 500
    sample_every: int = 5000

cfg = Config()
cfg


## 2. Données : CelebA

Images recadrées au centre puis redimensionnées en $64\times64$ RGB, et normalisées dans $[-1,1]$ pour s'accorder au `tanh` du générateur.

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms

transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

data_root = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba'

train_set = ImageFolder(root=data_root, transform=transform)
train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=2, drop_last=True, pin_memory=True)

def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch

data_iter = infinite_loader(train_loader)
print('Dataset size:', len(train_set))
x_demo, _ = next(data_iter)
print('Batch shape:', x_demo.shape, 'min/max:', x_demo.min().item(), x_demo.max().item())

## 3. Architectures : Générateur et Critique (DCGAN)

Le critique **n'est pas un classifieur** : il projette l'image dans $\mathbb{R}^{16}$ (les *features*).
La MMD est ensuite calculée sur ces features. On définit ici les **classes** ; les instances
seront créées **à chaque expérience** (réinitialisation propre).

In [ ]:
class Generator(nn.Module):
    """DCGAN generator: z (B, z_dim) -> image (B, 3, 64, 64) dans [-1, 1]."""
    def __init__(self, z_dim=100, ngf=64, channels=3):
        super().__init__()
        self.net = nn.Sequential(
            # z (B, z_dim, 1, 1) -> (B, ngf*8, 4, 4)
            nn.ConvTranspose2d(z_dim, ngf * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # -> (B, ngf*4, 8, 8)
            nn.ConvTranspose2d(ngf * 8, ngf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # -> (B, ngf*2, 16, 16)
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # -> (B, ngf, 32, 32)
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # -> (B, channels, 64, 64)
            nn.ConvTranspose2d(ngf, channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        if z.dim() == 2:
            z = z.view(z.size(0), z.size(1), 1, 1)
        return self.net(z)


class Critic(nn.Module):
    """DCGAN-style critic: image (B,3,64,64) -> features (B, feat_dim).

    Pas de BatchNorm dans le critique (recommandé par WGAN-GP et MMD-GAN, sinon le gradient
    penalty perd son sens : il faut une dépendance sample-par-sample). On utilise LayerNorm.
    """
    def __init__(self, channels=3, ndf=16, feat_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            # (B,3,64,64) -> (B, ndf, 32, 32)
            nn.Conv2d(channels, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (B, ndf*2, 16, 16)
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1),
            nn.LayerNorm([ndf * 2, 16, 16]),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (B, ndf*4, 8, 8)
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1),
            nn.LayerNorm([ndf * 4, 8, 8]),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (B, ndf*8, 4, 4)
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1),
            nn.LayerNorm([ndf * 8, 4, 4]),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.head = nn.Linear(ndf * 8 * 4 * 4, feat_dim)

    def forward(self, x):
        h = self.conv(x)
        h = h.flatten(1)
        return self.head(h)

## 4. Noyaux et estimateur MMD²

On définit les trois familles de noyaux comparées, puis un estimateur MMD² **générique**
qui prend n'importe quel noyau en argument.

In [ ]:
def _sqdist(x, y):
    """Distances euclidiennes au carré entre paires. x:(m,d), y:(n,d) -> (m,n)."""
    xx = (x * x).sum(1, keepdim=True)
    yy = (y * y).sum(1, keepdim=True).t()
    return (xx + yy - 2.0 * (x @ y.t())).clamp(min=0)


# ---- Noyau 1 : RBF fixe -------------------------------------------------------
def rbf_kernel(x, y, sigma=1.0):
    """k(x,y) = exp(-||x-y||^2 / (2 sigma^2))."""
    d2 = _sqdist(x, y)
    return torch.exp(-d2 / (2.0 * sigma ** 2))


# ---- Noyau 2 : RQ* fixe (rationnel quadratique + linéaire) --------------------
def rq_kernel(x, y, alphas=(0.2, 0.5, 1.0, 2.0, 5.0)):
    """Mélange de noyaux rationnels quadratiques (éq. 6 du papier)."""
    d2 = _sqdist(x, y)
    K = torch.zeros_like(d2)
    for a in alphas:
        K = K + (1.0 + d2 / (2.0 * a)) ** (-a)
    return K

def rq_dot_kernel(x, y, alphas=(0.2, 0.5, 1.0, 2.0, 5.0)):
    """Noyau rq* du papier : mélange RQ + terme linéaire."""
    return rq_kernel(x, y, alphas) + x @ y.t()


# ---- Noyau 3 : RQ* APPRIS (bandwidths alpha apprenables) ----------------------
class LearnableRQDotKernel(nn.Module):
    """Mélange rq* avec bandwidths alpha APPRIS.
    Paramétrisé par log(alpha) pour garantir alpha > 0 librement.
    """
    def __init__(self, init_alphas=(0.2, 0.5, 1.0, 2.0, 5.0), include_dot=True):
        super().__init__()
        log_alphas = torch.log(torch.tensor(init_alphas, dtype=torch.float32))
        self.log_alphas = nn.Parameter(log_alphas)
        self.include_dot = include_dot

    def alphas(self):
        # clamp pour empêcher explosion / effondrement : log(0.01)≈-4.6, log(100)≈4.6
        return self.log_alphas.clamp(-4.6, 4.6).exp()

    def forward(self, x, y):
        d2 = _sqdist(x, y)
        K = torch.zeros_like(d2)
        for a in self.alphas():
            K = K + (1.0 + d2 / (2.0 * a)) ** (-a)
        if self.include_dot:
            K = K + x @ y.t()
        return K


# ---- Estimateur MMD² sans biais GÉNÉRIQUE (éq. 4 du papier) -------------------
def mmd2_unbiased(f_x, f_y, kernel):
    """Estimateur sans biais de MMD^2 avec un noyau quelconque (callable kernel(x,y))."""
    m, n = f_x.size(0), f_y.size(0)
    assert m > 1 and n > 1, "L'estimateur sans biais nécessite m, n >= 2."
    k_xx = kernel(f_x, f_x)
    k_yy = kernel(f_y, f_y)
    k_xy = kernel(f_x, f_y)
    eye_m = torch.eye(m, dtype=torch.bool, device=f_x.device)
    eye_n = torch.eye(n, dtype=torch.bool, device=f_y.device)
    sum_xx = k_xx.masked_fill(eye_m, 0).sum() / (m * (m - 1))
    sum_yy = k_yy.masked_fill(eye_n, 0).sum() / (n * (n - 1))
    sum_xy = k_xy.sum() / (m * n)
    return sum_xx + sum_yy - 2.0 * sum_xy


# Test rapide : MMD ~ 0 si distributions identiques, > 0 sinon
with torch.no_grad():
    a = torch.randn(128, 16, device=device)
    b = torch.randn(128, 16, device=device)
    c = torch.randn(128, 16, device=device) + 3.0
    kfn = lambda x, y: rq_dot_kernel(x, y)
    print('MMD^2 (a, b) ~ 0 :', mmd2_unbiased(a, b, kfn).item())
    print('MMD^2 (a, c) > 0 :', mmd2_unbiased(a, c, kfn).item())


## 5. Fonction témoin et gradient penalty

Le gradient penalty contraint la norme du gradient de la **fonction témoin** à être proche de 1,
sur des points interpolés entre réels et faux. Les deux fonctions prennent le **noyau en argument**
pour fonctionner avec n'importe laquelle des trois variantes.

In [ ]:
def witness_function(x, real_feat, fake_feat, critic, kernel):
    """Witness empirique f_hat(x) aux points x (batch d'images)."""
    f_x = critic(x)
    k_real = kernel(real_feat, f_x).mean(dim=0)
    k_fake = kernel(fake_feat, f_x).mean(dim=0)
    return k_real - k_fake


def gradient_penalty(critic, real, fake, kernel, device):
    """Gradient penalty sur la fonction témoin, aux points interpolés."""
    batch_size = real.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interp = alpha * real + (1 - alpha) * fake
    interp.requires_grad_(True)

    with torch.no_grad():
        real_feat = critic(real)
        fake_feat = critic(fake)

    w = witness_function(interp, real_feat, fake_feat, critic, kernel)

    grads = torch.autograd.grad(
        outputs=w.sum(),
        inputs=interp,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    grad_norm = grads.view(batch_size, -1).norm(2, dim=1)
    return ((grad_norm - 1) ** 2).mean()


## 6. Fonction d'entraînement générique

`run_experiment` encapsule une expérience complète. Elle **réinitialise** G, D (et le noyau
appris) à chaque appel, puis entraîne avec les options choisies :

- `kernel_type` : `'rbf'`, `'rq'` ou `'learn'`
- `use_gp` : activer le gradient penalty
- `use_act` : activer la pénalité d'activation

Elle renvoie un dictionnaire avec l'historique, les échantillons et le générateur final.

In [ ]:
def show_grid(imgs, title=''):
    grid = utils.make_grid(imgs.detach().cpu(), nrow=8, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=(5, 5))
    plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    plt.title(title); plt.axis('off'); plt.show()


def run_experiment(name, kernel_type='rq', use_gp=True, use_act=True, seed=0, verbose=True):
    """Entraîne un MMD-GAN avec le noyau et la régularisation choisis. Renvoie un dict de résultats."""
    # --- Réinitialisation propre (équité entre expériences) -------------------
    torch.manual_seed(seed); np.random.seed(seed)
    G = Generator(cfg.z_dim, cfg.g_filters, cfg.channels).to(device)
    D = Critic(cfg.channels, cfg.d_filters, cfg.feat_dim).to(device)

    # --- Construction du noyau ------------------------------------------------
    kernel_module = None
    if kernel_type == 'rbf':
        kernel = lambda x, y: rbf_kernel(x, y, cfg.sigma)
    elif kernel_type == 'rq':
        kernel = lambda x, y: rq_dot_kernel(x, y)
    elif kernel_type == 'learn':
        kernel_module = LearnableRQDotKernel().to(device)
        kernel = kernel_module
    else:
        raise ValueError(kernel_type)

    # --- Optimiseurs ----------------------------------------------------------
    opt_G = torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    d_params = list(D.parameters())
    if kernel_module is not None:                 # le noyau appris s'optimise AVEC le critique
        d_params += list(kernel_module.parameters())
    opt_D = torch.optim.Adam(d_params, lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

    z_fixed = torch.randn(64, cfg.z_dim, device=device)
    history = {'iter': [], 'mmd2': [], 'd_loss': [], 'g_loss': [], 'gp': [], 'act': []}
    samples_log = []

    if verbose:
        print(f'\n=== {name}  (kernel={kernel_type}, GP={use_gp}, act={use_act}) ===')
    start = time.time()

    for it in range(1, cfg.n_iters + 1):
        # ---- 1) Critique --------------------------------------------------
        for _ in range(cfg.n_critic):
            real, _ = next(data_iter)
            real = real.to(device, non_blocking=True)
            bs = real.size(0)
            z = torch.randn(bs, cfg.z_dim, device=device)
            with torch.no_grad():
                fake = G(z)

            f_real = D(real)
            f_fake = D(fake)
            mmd2 = mmd2_unbiased(f_real, f_fake, kernel)

            d_loss = -mmd2
            gp_val = 0.0
            act_val = 0.0
            gp = gradient_penalty(D, real, fake, kernel, device)
            gp_val = gp.item()
            if use_gp:
                d_loss = d_loss + cfg.lambda_gp * gp
            act_penalty = (f_real ** 2).mean() + (f_fake ** 2).mean()
            act_val = act_penalty.item()
            if use_act:
                d_loss = d_loss + cfg.lambda_act * act_penalty
                

            opt_D.zero_grad(set_to_none=True)
            d_loss.backward()
            opt_D.step()

        # ---- 2) Générateur ------------------------------------------------
        z = torch.randn(cfg.batch_size, cfg.z_dim, device=device)
        fake = G(z)
        real, _ = next(data_iter)
        real = real.to(device, non_blocking=True)
        f_real = D(real)
        f_fake = D(fake)
        g_loss = mmd2_unbiased(f_real, f_fake, kernel)

        opt_G.zero_grad(set_to_none=True)
        g_loss.backward()
        opt_G.step()

        # ---- Logging ------------------------------------------------------
        if it % cfg.log_every == 0 or it == 1:
            history['iter'].append(it)
            history['mmd2'].append(mmd2.item())
            history['d_loss'].append(d_loss.item())
            history['g_loss'].append(g_loss.item())
            history['gp'].append(gp_val)
            history['act'].append(act_val)
            if verbose:
                msg = (f'[{it:6d}/{cfg.n_iters}] MMD^2={mmd2.item():+.4f} '
                       f'GP={gp_val:.3f} act={act_val:.3f} '
                       f'G={g_loss.item():+.4f} ({time.time()-start:.0f}s)')
                if kernel_module is not None:
                    msg += f"  alphas={np.round(kernel_module.alphas().detach().cpu().numpy(),2)}"
                print(msg)

        if it % cfg.sample_every == 0 or it == cfg.n_iters:
            G.eval()
            with torch.no_grad():
                samples = G(z_fixed)
            G.train()
            samples_log.append((it, samples.cpu()))

    if verbose:
        print(f'--- terminé en {time.time()-start:.0f}s ---')

    return {
        'name': name, 'kernel_type': kernel_type, 'use_gp': use_gp, 'use_act': use_act,
        'history': history, 'samples_log': samples_log,
        'G': G, 'D': D, 'kernel_module': kernel_module, 'z_fixed': z_fixed,
    }


## 7. Lancement des 9 expériences

⚠️ **Temps de calcul** : 9 × `n_iters`. Pour un premier test, réduis `cfg.n_iters` (ex. 3000–5000)
dans la cellule de config, puis relance tout. Sur GPU avec 50000 itérations, prévois plusieurs heures.

Chaque résultat est stocké dans le dictionnaire `RESULTS`.

In [ ]:
RESULTS = {}

experiments = [
    # (nom court, kernel_type, use_gp, use_act)
    ('rbf_plain',   'rbf',   False, False),
    ('rbf_gp',      'rbf',   True,  False),
    ('rbf_gp_act',  'rbf',   True,  True),
    ('rq_plain',    'rq',    False, False),
    ('rq_gp',       'rq',    True,  False)

]

for name, ktype, gp, act in experiments:
    RESULTS[name] = run_experiment(name, kernel_type=ktype, use_gp=gp, use_act=act)


## 8. Comparaison — échantillons finaux des 9 expériences

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for ax, (name, _, _, _) in zip(axes.flat, experiments):
    res = RESULTS[name]
    it_last, imgs = res['samples_log'][-1]
    grid = utils.make_grid(imgs[:25], nrow=5, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.set_title(name, fontsize=11); ax.axis('off')
plt.suptitle('Échantillons finaux par expérience', fontsize=14)
plt.tight_layout(); plt.show()


## 9. Comparaison — courbes d'apprentissage (MMD² et gradient penalty)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, _, _, _ in experiments:
    h = RESULTS[name]['history']
    axes[0].plot(h['iter'], h['mmd2'], label=name, alpha=0.8)
axes[0].set_title('MMD² (mesurée par le critique)')
axes[0].set_xlabel('itération'); axes[0].set_ylabel('MMD²')
axes[0].legend(fontsize=8); axes[0].grid(True)

for name, _, use_gp, _ in experiments:
    if use_gp:
        h = RESULTS[name]['history']
        axes[1].plot(h['iter'], h['gp'], label=name, alpha=0.8)
axes[1].set_title('Gradient penalty (cible ≈ 0)')
axes[1].set_xlabel('itération'); axes[1].set_ylabel('GP')
axes[1].legend(fontsize=8); axes[1].grid(True)

plt.tight_layout(); plt.show()


## 10. Sauvegarde des modèles et historiques

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
for name, _, _, _ in experiments:
    res = RESULTS[name]
    ckpt = {
        'G': res['G'].state_dict(),
        'D': res['D'].state_dict(),
        'history': res['history'],
        'config': cfg.__dict__,
    }
    if res['kernel_module'] is not None:
        ckpt['kernel'] = res['kernel_module'].state_dict()
    torch.save(ckpt, f'checkpoints/mmd_gan_{name}.pt')
print('Modèles sauvegardés dans checkpoints/')
